# Sequence Timeseries Extension Stability

Ce notebook reprend le script `sequence_timeseries_extension_stability.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Valide la stabilite des architectures temporelles candidates live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Repeated-split stability for additional time-series sequence architectures.
- Commande de reproduction referencee : time-series extension repeated stability, extension stability on original extra-10 seeds.
- Artefacts controles : Repeated-split stability for additional time-series architectures exists. (`runs/exp_041_timeseries_extension_stability/metrics/timeseries_extension_stability_summary.csv`); Extension stability run on original extra-10 seeds exists. (`runs/exp_043_timeseries_extension_stability_original_extra10/metrics/timeseries_extension_stability_summary.csv`).
- Run par defaut : `runs/exp_041_timeseries_extension_stability`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "sequence_timeseries_extension_stability.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch

from ml_pipeline import ROOT, write_json
from sequence_experiments import append_report, evaluate_catalogue_model, make_run_dir
from sequence_stability_experiments import normalize_for_split, selection_score, stratified_video_split
from sequence_timeseries_arch_extension import ModelSpec, load_sequence_dataset, train_one


DEFAULT_SEEDS = [
    101,
    202,
    303,
    404,
    505,
    606,
    707,
    808,
    909,
    1001,
    1111,
    1212,
    1313,
    1414,
    1515,
    1616,
    1717,
    1818,
    1919,
    2020,
]


## Fonction `default_specs`

Cette cellule definit `default_specs`. Elle prepare une partie du script.

In [ ]:
def default_specs():
    return [
        ModelSpec("resnet1d_aug_focal", "resnet1d", "focal", True),
        ModelSpec("inceptiontime_aug_focal", "inceptiontime", "focal", True),
        ModelSpec("separable_tcn_aug_focal", "separable_tcn", "focal", True),
        ModelSpec("bigru_attention_aug_focal", "bigru_attention", "focal", True),
        ModelSpec("conv_transformer_aug_focal", "conv_transformer", "focal", True),
        ModelSpec("patch_transformer_aug_focal", "patch_transformer", "focal", True),
    ]


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(metrics):
    h1 = metrics[metrics["horizon_s"].astype(float).eq(1.0)].copy()
    rows = []
    for (base_arch, split_name), group in h1.groupby(["base_architecture", "split"]):
        rows.append(
            {
                "base_architecture": base_arch,
                "split": split_name,
                "n_repeats": int(group["repeat_seed"].nunique()),
                "ap_mean": float(group["average_precision"].mean()),
                "ap_std": float(group["average_precision"].std(ddof=0)),
                "roc_auc_mean": float(group["roc_auc"].mean()),
                "roc_auc_std": float(group["roc_auc"].std(ddof=0)),
                "hit_rate_mean": float(group["best_hit_rate"].mean()),
                "hit_rate_std": float(group["best_hit_rate"].std(ddof=0)),
                "false_alarms_per_min_mean": float(group["best_false_alarms_per_min"].mean()),
                "false_alarms_per_min_std": float(group["best_false_alarms_per_min"].std(ddof=0)),
                "precision_mean": float(group["best_window_precision"].mean()),
                "precision_std": float(group["best_window_precision"].std(ddof=0)),
                "inference_ms_per_window_mean": float(group["inference_ms_per_window"].mean()),
                "model_size_mb_mean": float(group["model_size_bytes"].mean() / (1024 * 1024)),
            }
        )
    return pd.DataFrame(rows)


## Fonction `write_summary`

Cette cellule definit `write_summary`. Elle prepare une partie du script.

In [ ]:
def write_summary(run_dir, summary, specs, seeds):
    val = summary[summary["split"] == "val"].copy()
    val["stability_score"] = (
        val["ap_mean"]
        + 0.5 * val["hit_rate_mean"]
        + 0.2 * val["precision_mean"]
        - 0.03 * val["false_alarms_per_min_mean"].clip(upper=20)
    )
    val = val.sort_values("stability_score", ascending=False)
    test = summary[summary["split"] == "test"].sort_values("ap_mean", ascending=False)

    lines = ["# Repeated-Split Time-Series Architecture Extension Stability", ""]
    lines.append("This repeats the added time-series architectures across parent-video splits. Each repeat re-normalizes sequence features from that repeat's train videos only.")
    lines.append("")
    lines.append(f"- Repeats: `{len(seeds)}`")
    lines.append(f"- Seeds: `{seeds}`")
    lines.append(f"- Specs: `{[spec.name for spec in specs]}`")
    lines.append("- Split policy: parent-video split first, then all sequence windows inherit the parent split.")
    lines.append("")
    lines.append("## Validation Ranking")
    lines.append("")
    lines.append("| rank | architecture | AP mean | AP std | hit | FA/min | precision | ms/window |")
    lines.append("|---:|---|---:|---:|---:|---:|---:|---:|")
    for rank, (_, row) in enumerate(val.iterrows(), start=1):
        lines.append(
            f"| {rank} | {row['base_architecture']} | {row['ap_mean']:.3f} | {row['ap_std']:.3f} | "
            f"{row['hit_rate_mean']:.3f} | {row['false_alarms_per_min_mean']:.3f} | "
            f"{row['precision_mean']:.3f} | {row['inference_ms_per_window_mean']:.4f} |"
        )
    lines.append("")
    lines.append("## Test Diagnostics")
    lines.append("")
    lines.append("| rank | architecture | AP mean | AP std | ROC AUC | hit | FA/min | precision | ms/window |")
    lines.append("|---:|---|---:|---:|---:|---:|---:|---:|---:|")
    for rank, (_, row) in enumerate(test.iterrows(), start=1):
        lines.append(
            f"| {rank} | {row['base_architecture']} | {row['ap_mean']:.3f} | {row['ap_std']:.3f} | "
            f"{row['roc_auc_mean']:.3f} | {row['hit_rate_mean']:.3f} | "
            f"{row['false_alarms_per_min_mean']:.3f} | {row['precision_mean']:.3f} | "
            f"{row['inference_ms_per_window_mean']:.4f} |"
        )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- These results put the new architecture families under the same repeated parent-video split protocol as the earlier sequence shortlist.")
    lines.append("- If a fixed-split winner drops under repeated splits, it should not replace the TCN-family conclusion.")
    lines.append("- If an extension family stays competitive, it should be retained as a serious baseline and candidate ensemble member.")
    summary_path = run_dir / "timeseries_extension_stability_summary.md"
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return summary_path


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    run_dir = make_run_dir(args.run_name)
    source_run, X_norm, y, base_meta = load_sequence_dataset(args.sequence_run)
    data = np.load(source_run / "features" / "sequence_dataset.npz")
    mean = data["mean"].astype(np.float32)
    std = data["std"].astype(np.float32)
    X_raw = X_norm * std.reshape(1, 1, -1) + mean.reshape(1, 1, -1)
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    specs = default_specs()
    if args.quick:
        specs = specs[:2]

    write_json(
        run_dir / "metrics" / "timeseries_extension_stability_config.json",
        {
            "sequence_run": str(source_run),
            "seeds": args.seeds,
            "rows": int(len(base_meta)),
            "seq_len": int(X_norm.shape[1]),
            "feature_count": int(X_norm.shape[2]),
            "device": str(device),
            "specs": [spec.__dict__ for spec in specs],
            "split_policy": "repeated parent-video split; sequence windows inherit split",
            "normalization": "train split mean/std recomputed per repeat",
        },
    )

    all_metrics = []
    all_history = []
    split_rows = []
    for seed in args.seeds:
        split = stratified_video_split(base_meta, seed)
        meta = base_meta.copy()
        meta["split"] = meta["video_id"].map(split)
        X, split_mean, split_std = normalize_for_split(X_raw, meta)
        meta.to_csv(run_dir / "features" / f"split_seed_{seed}.csv", index=False)
        np.savez_compressed(run_dir / "features" / f"normalizer_seed_{seed}.npz", mean=split_mean, std=split_std)
        split_rows.append({"seed": seed, **meta.groupby("split")["video_id"].nunique().to_dict()})

        for base_spec in specs:
            spec = ModelSpec(
                name=f"seed{seed}_{base_spec.name}",
                kind=base_spec.kind,
                loss=base_spec.loss,
                augment=base_spec.augment,
            )
            print(f"training {spec.name} ({spec.kind}, {spec.loss})")
            model_args = SimpleNamespace(
                seed=seed,
                batch_size=args.batch_size,
                lr=args.lr,
                weight_decay=args.weight_decay,
                epochs=args.epochs,
                patience=args.patience,
                label_smoothing=args.label_smoothing,
                focal_gamma=args.focal_gamma,
                grad_clip=args.grad_clip,
            )
            model, history, train_time_s, model_size_bytes = train_one(spec, X, y, meta, run_dir, model_args, device)
            for row in history:
                row["repeat_seed"] = seed
                row["base_architecture"] = base_spec.name
            all_history.extend(history)
            rows, _ = evaluate_catalogue_model(
                spec.name,
                model,
                X,
                y,
                meta,
                run_dir,
                device,
                train_time_s,
                model_size_bytes,
                args.batch_size,
                spec.loss,
            )
            for row in rows:
                row["repeat_seed"] = seed
                row["base_architecture"] = base_spec.name
                row["selection_score"] = selection_score(row) if row["split"] == "val" and float(row["horizon_s"]) == 1.0 else np.nan
            all_metrics.extend(rows)
            pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "timeseries_extension_stability_metrics.csv", index=False)
            pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "timeseries_extension_stability_training_history.csv", index=False)

    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "timeseries_extension_stability_metrics.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "timeseries_extension_stability_training_history.csv", index=False)
    pd.DataFrame(split_rows).to_csv(run_dir / "metrics" / "timeseries_extension_stability_split_counts.csv", index=False)
    summary = summarize(metrics)
    summary.to_csv(run_dir / "metrics" / "timeseries_extension_stability_summary.csv", index=False)
    summary_path = write_summary(run_dir, summary, specs, args.seeds)
    append_report(
        run_dir,
        "Repeated-Split Time-Series Extension Stability",
        f"- Summary: `{summary_path}`\n- Metrics: `{run_dir / 'metrics' / 'timeseries_extension_stability_summary.csv'}`",
    )
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Repeated-split stability for additional time-series sequence architectures.")
    parser.add_argument("--sequence-run", default="runs/exp_008_sequence_len60_catalogue")
    parser.add_argument("--run-name", default="exp_041_timeseries_extension_stability")
    parser.add_argument("--seeds", nargs="+", type=int, default=DEFAULT_SEEDS)
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--patience", type=int, default=4)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=8e-4)
    parser.add_argument("--weight-decay", type=float, default=1.5e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.05)
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--grad-clip", type=float, default=3.0)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--quick", action="store_true")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_041_timeseries_extension_stability_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["sequence_timeseries_extension_stability.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
